In [1]:
import numpy as np
import pandas as pd

dataset_path = "/content/sample_data/mnist_train_small.csv"

In [2]:
def relu(X):
  return np.maximum(0, X)
def sigmoid(X):
  return 1 / (1 + np.exp(-1 * X))
def softmax(X):
  X = X - np.max(X, axis=0, keepdims=True)
  return np.exp(X) / np.sum(np.exp(X), axis=0, keepdims=True)

def deriv_relu(X):
  return (X > 0).astype(int)

## Data

Load the MNIST CSV, shuffle the examples, and split them into training, validation, and test sets. Pixel values are normalized to the range [0, 1].


In [3]:
df = pd.read_csv(dataset_path)
df

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19994,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19995,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19996,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19997,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
X = np.array([
    [1,2,3,4,5],
    [5,6,7,8,9]
]).T
W = np.array([
    [0.1,0.2,0.3,0.4,0.5],
    [0.1,0.2,0.3,0.4,0.5],
    [0.1,0.2,0.3,0.4,0.5]
])
print(X.shape)
print(W.shape)
Z = W @ X
print(Z.shape)

(5, 2)
(3, 5)
(3, 2)


In [5]:
data = np.array(df)
m, n = data.shape

np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 255.0

data_test = data[1000:2000].T
Y_test = data_test[0]
X_test = data_test[1:n] / 255.0

data_train = data[2000:].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255.0

In [6]:
Y_train.shape

(17999,)

## Model

Two-layer fully connected classifier: 784 input features, 10 ReLU hidden units, and 10 softmax output probabilities. Weights are initialized with positive uniform random values scaled by the He factor $\sqrt{2/n_{\mathrm{in}}}$; biases start at zero. Training uses one-hot targets, cross-entropy/softmax gradients, and full-batch gradient descent.


In [7]:
def init_params():
  W1 = np.random.rand(10,784) * np.sqrt(2 / 784)
  W2 = np.random.rand(10,10) * np.sqrt(2 / 10)
  b1 = np.zeros((10,1))
  b2 = np.zeros((10,1))
  return W1,b1,W2,b2

def forward_prop(W1, b1, W2, b2, X):
  Z1 = W1 @ X + b1
  A1 = relu(Z1)

  Z2 = W2 @ A1 + b2
  A2 = softmax(Z2)
  return Z1,A1,Z2,A2
def one_hot(Y):
  one_hot_y = np.zeros((Y.size, Y.max() + 1))
  one_hot_y[np.arange(Y.size), Y] = 1
  one_hot_y = one_hot_y.T
  return one_hot_y

def back_prop(Z1, A1, Z2, A2, W2, X, Y):
  m = Y.size


  one_hot_y = one_hot(Y)

  dZ2 = A2 - one_hot_y
  dZ1 = deriv_relu(Z1) * (W2.T @ dZ2)

  dW2 = (dZ2 @ A1.T) / m
  db2 = np.sum(dZ2, axis=1, keepdims=True) / m

  dW1 = (dZ1 @ X.T) / m
  db1 = np.sum(dZ1, axis=1, keepdims=True) / m

  return dW1, dW2, db1, db2

def update_params(W1, W2, b1, b2, dW1, dW2, db1, db2, alpha):
  W1 = W1 - alpha * dW1
  W2 = W2 - alpha * dW2

  b1 = b1 - alpha * db1
  b2 = b2 - alpha * db2

  return W1, W2, b1, b2

In [8]:
def get_prediciton(A2):
  return np.argmax(A2, axis=0)

def get_accuracy(predictions, Y):
  return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, X_val, Y_val, epochs, alpha):
  W1, b1, W2, b2 = init_params()

  for epoch in range(epochs):

    Z1,A1,Z2,A2 = forward_prop(W1, b1, W2, b2, X)
    dW1, dW2, db1, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
    W1, W2, b1, b2 = update_params(W1, W2, b1, b2, dW1, dW2, db1, db2, alpha)



    if epoch % 50 == 0:
      print("Epoch:", epoch)
      print("Accuracy:", get_accuracy(get_prediciton(A2), Y))
      Z1,A1,Z2,A2 = forward_prop(W1, b1, W2, b2, X_val)
      print("Validation Accuracy:", get_accuracy(get_prediciton(A2), Y_val))

  return W1, b1, W2, b2

In [9]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, X_dev, Y_dev, 500, 0.1)

Epoch: 0
Accuracy: 0.10067225956997611
Validation Accuracy: 0.112
Epoch: 50
Accuracy: 0.7429301627868214
Validation Accuracy: 0.744
Epoch: 100
Accuracy: 0.8414911939552198
Validation Accuracy: 0.831
Epoch: 150
Accuracy: 0.8723262403466859
Validation Accuracy: 0.861
Epoch: 200
Accuracy: 0.8846602589032724
Validation Accuracy: 0.882
Epoch: 250
Accuracy: 0.8926051447302628
Validation Accuracy: 0.888
Epoch: 300
Accuracy: 0.8974387465970332
Validation Accuracy: 0.892
Epoch: 350
Accuracy: 0.9018278793266292
Validation Accuracy: 0.896
Epoch: 400
Accuracy: 0.9045502527918218
Validation Accuracy: 0.897
Epoch: 450
Accuracy: 0.9069948330462804
Validation Accuracy: 0.899


## Test accuracy

Evaluate the trained model on the held-out test set without updating parameters.


In [10]:
_, _, _, A2_test = forward_prop(W1, b1, W2, b2, X_test)
test_predictions = get_prediciton(A2_test)
print("Test Accuracy:", get_accuracy(test_predictions, Y_test))


Test Accuracy: 0.898
